In [1]:
import pandas as pd
import numpy as np
import random
import datetime as dt
from collections import Counter

print("Environment loaded successfully")

Environment loaded successfully


In [2]:
def simulate_pcap_dataframe(n_packets=5000):
    rows = []
    base_time = dt.datetime(2026, 1, 31, 10, 0, 0)
    
    for i in range(n_packets):
        ts = base_time + dt.timedelta(seconds=i/100.0)
        src = f"192.168.1.{random.randint(2, 254)}"
        dst = f"192.168.1.{random.randint(2, 254)}"
        proto = random.choice(["TCP", "UDP", "ARP"])
        flags = ""
        sport = random.randint(1024, 65535)
        dport = random.choice([22, 80, 443, 3389])
        src_mac = None

        if random.random() < 0.08 and src == "192.168.1.50":
            proto = "TCP"
            flags = "S"
            dport = 80

        if random.random() < 0.03:
            proto = "ARP"
            src_mac_options = ["aa:bb:cc:dd:ee:ff", "11:22:33:44:55:66", "99:88:77:66:55:44"]
            src_mac = random.choice(src_mac_options)

        rows.append({
            "timestamp": ts,
            "src_ip": src,
            "dst_ip": dst,
            "protocol": proto,
            "flags": flags,
            "src_port": sport,
            "dst_port": dport,
            "src_mac": src_mac
        })
    
    return pd.DataFrame(rows)

df_packets = simulate_pcap_dataframe(5000)
print(f"Generated {len(df_packets)} packets")
df_packets.head()


Generated 5000 packets


,timestamp,src_ip,dst_ip,protocol,flags,src_port,dst_port,src_mac
0,2026-01-31 10:00:00.000,192.168.1.115,192.168.1.3,TCP,,9056,80,None
1,2026-01-31 10:00:00.010,192.168.1.61,192.168.1.133,TCP,,43402,3389,None
2,2026-01-31 10:00:00.020,192.168.1.109,192.168.1.199,ARP,,25337,443,None
3,2026-01-31 10:00:00.030,192.168.1.60,192.168.1.228,UDP,,7662,22,None
4,2026-01-31 10:00:00.040,192.168.1.32,192.168.1.65,ARP,,20380,443,None


In [3]:
def detect_syn_flood(df, threshold=25):
    df_tcp = df[df['protocol'] == 'TCP'].copy()
    df_syn = df_tcp[df_tcp['flags'].str.contains('S', na=False)].copy()
    df_syn['timestamp'] = pd.to_datetime(df_syn['timestamp'])
    df_syn['time_bucket'] = df_syn['timestamp'].dt.floor('1S')
    
    syn_counts = df_syn.groupby(['time_bucket', 'src_ip', 'dst_port']).size().reset_index(name='syn_count')
    suspicious = syn_counts[syn_counts['syn_count'] >= threshold].sort_values('syn_count', ascending=False)
    return suspicious

suspicious_syn = detect_syn_flood(df_packets, threshold=25)
print("SYN Flood Results:")
print(suspicious_syn.head())

SYN Flood Results:
Empty DataFrame
Columns: [time_bucket, src_ip, dst_port, syn_count]
Index: []


In [4]:
def detect_arp_spoof(df):
    df_arp = df[df['protocol'] == 'ARP'].copy()
    
    ip_to_macs = df_arp.groupby('src_ip')['src_mac'].nunique().reset_index()
    ip_to_macs.columns = ['src_ip', 'unique_macs']
    
    mac_to_ips = df_arp.groupby('src_mac')['src_ip'].nunique().reset_index()
    mac_to_ips.columns = ['src_mac', 'unique_ips']
    
    suspicious_ips = ip_to_macs[ip_to_macs['unique_macs'] > 1]
    suspicious_macs = mac_to_ips[mac_to_ips['unique_ips'] > 1]
    
    return suspicious_ips, suspicious_macs

susp_ips, susp_macs = detect_arp_spoof(df_packets)
print("ARP Spoofing Results:")
print("Suspicious IPs:", susp_ips)
print("Suspicious MACs:", susp_macs)

ARP Spoofing Results:
Suspicious IPs:             src_ip  unique_macs
21   192.168.1.119            2
27   192.168.1.124            2
30   192.168.1.127            3
31   192.168.1.128            2
33    192.168.1.13            2
40   192.168.1.136            2
55    192.168.1.15            2
65   192.168.1.159            2
70   192.168.1.163            2
83   192.168.1.175            2
100  192.168.1.190            2
109  192.168.1.199            2
117  192.168.1.206            2
126  192.168.1.214            2
131  192.168.1.219            2
158  192.168.1.243            2
159  192.168.1.244            2
173   192.168.1.28            2
176   192.168.1.30            2
206   192.168.1.58            2
218   192.168.1.69            2
239   192.168.1.88            2
244   192.168.1.92            2
Suspicious MACs:              src_mac  unique_ips
0  11:22:33:44:55:66          47
1  99:88:77:66:55:44          56
2  aa:bb:cc:dd:ee:ff          38


In [7]:
def run_analysis():
    print("NETWORK THREAT HUNTING REPORT")
    print("=" * 40)
    
    syn_results = detect_syn_flood(df_packets, 25)
    if not syn_results.empty:
        print("SYN FLOOD DETECTED:")
        for _, row in syn_results.head(3).iterrows():
            print(f"  {row['src_ip']} -> port {row['dst_port']}: {row['syn_count']} SYNs")
    else:
        print("No SYN flood detected")
    
    arp_ips, arp_macs = detect_arp_spoof(df_packets)
    if not arp_ips.empty:
        print("\nARP SPOOFING DETECTED:")
        for _, row in arp_ips.iterrows():
            print(f"  IP {row['src_ip']} -> {row['unique_macs']} MACs")
    else:
        print("\nNo ARP spoofing detected")
    
    print("\nANALYSIS COMPLETE")

run_analysis()

NETWORK THREAT HUNTING REPORT
No SYN flood detected

ARP SPOOFING DETECTED:
  IP 192.168.1.119 -> 2 MACs
  IP 192.168.1.124 -> 2 MACs
  IP 192.168.1.127 -> 3 MACs
  IP 192.168.1.128 -> 2 MACs
  IP 192.168.1.13 -> 2 MACs
  IP 192.168.1.136 -> 2 MACs
  IP 192.168.1.15 -> 2 MACs
  IP 192.168.1.159 -> 2 MACs
  IP 192.168.1.163 -> 2 MACs
  IP 192.168.1.175 -> 2 MACs
  IP 192.168.1.190 -> 2 MACs
  IP 192.168.1.199 -> 2 MACs
  IP 192.168.1.206 -> 2 MACs
  IP 192.168.1.214 -> 2 MACs
  IP 192.168.1.219 -> 2 MACs
  IP 192.168.1.243 -> 2 MACs
  IP 192.168.1.244 -> 2 MACs
  IP 192.168.1.28 -> 2 MACs
  IP 192.168.1.30 -> 2 MACs
  IP 192.168.1.58 -> 2 MACs
  IP 192.168.1.69 -> 2 MACs
  IP 192.168.1.88 -> 2 MACs
  IP 192.168.1.92 -> 2 MACs

ANALYSIS COMPLETE
